In [1]:
%load_ext watermark

In [2]:
%watermark

Last updated: 2025-10-14T00:29:37.212662+00:00

Python implementation: CPython
Python version       : 3.13.7
IPython version      : 9.6.0

Compiler    : GCC 14.3.0
OS          : Linux
Release     : 6.11.0-1016-nvidia
Machine     : aarch64
Processor   : aarch64
CPU cores   : 20
Architecture: 64bit



In [3]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
import optuna
import gc
import logging

/opt/conda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
%watermark --iversions

sklearn: 1.7.2
logging: 0.5.1.2
optuna : 4.5.0
numpy  : 2.2.6
pandas : 2.3.3
xgboost: 3.0.5



In [5]:
%%time
train_folds = []
val_folds = []
train_ys = []
val_ys = []

for i in range(5):
    print(f'Loading fold {i}')
    train_fold = pd.read_csv(f'../input/xgtrain_fold_{i}_l.csv.gz')

    
    val_fold = pd.read_csv(f'../input/xgval_fold_{i}_l.csv.gz')

    
    
    train_y = train_fold['target']
    train_fold = train_fold[train_fold.columns.difference(['target'])]
    
    val_y = val_fold['target']
    val_fold = val_fold[val_fold.columns.difference(['target'])]
    
    train_folds.append(train_fold)
    val_folds.append(val_fold)
    
    train_ys.append(train_y)
    val_ys.append(val_y)

Loading fold 0
Loading fold 1
Loading fold 2
Loading fold 3
Loading fold 4
CPU times: user 1.89 s, sys: 191 ms, total: 2.08 s
Wall time: 2.08 s


In [6]:
train = pd.read_csv('../input/train.csv.zip')

shift = 200

target0 = train['loss'].values
target = np.log(target0+shift)

In [7]:
train_oof = np.zeros((target.shape[0],))

num_round = 1000

def objective(trial):
        
    params = {
        'objective': 'reg:squarederror',
        'device':'cuda',
        'base_score':7.76,
        'tree_method':'hist',  # 'gpu_hist','hist'
        'lambda': trial.suggest_float('lambda',1e-3,10.0, log=True),
        'alpha': trial.suggest_float('alpha',1e-3,10.0, log=True),
        'gamma': trial.suggest_float('gamma',1e-3,10.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3,1.0),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'learning_rate': trial.suggest_float('learning_rate', 0.001,0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 25),
        'min_child_weight': trial.suggest_int('min_child_weight', 1,300),
        'eval_metric': trial.suggest_categorical('eval_metric',['rmse']),

    }

    kf = KFold(5, shuffle=True, random_state=137)

    for i, (train_index, val_index) in enumerate(kf.split(train,target)):
        dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
        dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
        output = xgb.train(params, dtrain, num_round)
        #booster = output['booster']  # booster is the trained model
        #booster.set_param({'predictor': 'gpu_predictor'})
        predictions = output.predict(dval)
        train_oof[val_index] = np.exp(predictions) - shift
        del dtrain, dval, output
        gc.collect()
        gc.collect()

    mae = mean_absolute_error(target0, train_oof)
    
    return mae

In [8]:
logger = logging.getLogger()
logger.setLevel(logging.INFO)  # Setup the root logger.
logger.addHandler(logging.FileHandler("optuna_xgb_output_l_4_DGX_Spark_cuda.log", mode="w"))

optuna.logging.enable_propagation()  # Propagate logs to the root logger.
optuna.logging.disable_default_handler()  # Stop showing logs in sys.stderr.

study = optuna.create_study(storage="sqlite:///xgb_optuna_allstate_l_4_DGX_Spark_cuda.db", study_name="five_fold_optuna_xgb_l_4", direction='minimize')

In [9]:
%%time
logger.info("Start optimization.")
study.optimize(objective, n_trials=3)

CPU times: user 42.3 s, sys: 15.1 s, total: 57.4 s
Wall time: 2min 29s


In [10]:
df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
df.head()

,number,value,params_alpha,params_colsample_bytree,params_eval_metric,params_gamma,params_lambda,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
0,0,1163.750462,0.004873,0.507590,rmse,0.004490,0.002372,0.040535,3,136,0.794685,COMPLETE
1,1,1142.008138,5.069154,0.563836,rmse,0.032892,0.014103,0.017040,8,244,0.871797,COMPLETE
2,2,1188.984783,2.073778,0.492736,rmse,8.479396,0.062394,0.006109,19,147,0.463447,COMPLETE


In [11]:
%%time
study.optimize(objective, n_trials=5)
df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
df.to_csv('optuna_xgb_output_l_4_DGX_Spark_cuda.csv', index=False)
df

CPU times: user 2min 49s, sys: 51 s, total: 3min 40s
Wall time: 11min 24s


,number,value,params_alpha,params_colsample_bytree,params_eval_metric,params_gamma,params_lambda,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
0,0,1163.750462,0.004873,0.507590,rmse,0.004490,0.002372,0.040535,3,136,0.794685,COMPLETE
1,1,1142.008138,5.069154,0.563836,rmse,0.032892,0.014103,0.017040,8,244,0.871797,COMPLETE
2,2,1188.984783,2.073778,0.492736,rmse,8.479396,0.062394,0.006109,19,147,0.463447,COMPLETE
3,3,1318.024755,0.015029,0.764154,rmse,1.176339,0.026305,0.001382,13,145,0.820977,COMPLETE
4,4,1156.852653,0.340799,0.448980,rmse,3.603275,0.536545,0.020737,7,252,0.870824,COMPLETE
5,5,1146.824971,0.002893,0.354340,rmse,0.047990,0.096273,0.031926,23,186,0.547834,COMPLETE
6,6,1139.460243,0.025158,0.490851,rmse,0.060088,0.530663,0.013811,23,82,0.811702,COMPLETE
7,7,1316.413098,1.427005,0.955906,rmse,0.003368,0.029416,0.003541,3,291,0.633825,COMPLETE


In [12]:
%%time
study.optimize(objective, n_trials=100)
df = study.trials_dataframe(attrs=('number', 'value', 'params', 'state'))
df.to_csv('optuna_xgb_output_l_4_DGX_Spark.csv', index=False)
df.head(20)

CPU times: user 57min 18s, sys: 16min 30s, total: 1h 13min 49s
Wall time: 3h 53min 39s


,number,value,params_alpha,params_colsample_bytree,params_eval_metric,params_gamma,params_lambda,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
0,0,1163.750462,0.004873,0.507590,rmse,0.004490,0.002372,0.040535,3,136,0.794685,COMPLETE
1,1,1142.008138,5.069154,0.563836,rmse,0.032892,0.014103,0.017040,8,244,0.871797,COMPLETE
2,2,1188.984783,2.073778,0.492736,rmse,8.479396,0.062394,0.006109,19,147,0.463447,COMPLETE
3,3,1318.024755,0.015029,0.764154,rmse,1.176339,0.026305,0.001382,13,145,0.820977,COMPLETE
4,4,1156.852653,0.340799,0.448980,rmse,3.603275,0.536545,0.020737,7,252,0.870824,COMPLETE
5,5,1146.824971,0.002893,0.354340,rmse,0.047990,0.096273,0.031926,23,186,0.547834,COMPLETE
6,6,1139.460243,0.025158,0.490851,rmse,0.060088,0.530663,0.013811,23,82,0.811702,COMPLETE
7,7,1316.413098,1.427005,0.955906,rmse,0.003368,0.029416,0.003541,3,291,0.633825,COMPLETE
8,8,1237.589780,0.030456,0.596654,rmse,2.358296,0.060091,0.002373,21,250,0.503027,COMPLETE
9,9,1319.781271,4.871841,0.491466,rmse,5.258412,3.838830,0.001612,19,222,0.961964,COMPLETE


In [13]:
df

,number,value,params_alpha,params_colsample_bytree,params_eval_metric,params_gamma,params_lambda,params_learning_rate,params_max_depth,params_min_child_weight,params_subsample,state
0,0,1163.750462,0.004873,0.507590,rmse,0.004490,0.002372,0.040535,3,136,0.794685,COMPLETE
1,1,1142.008138,5.069154,0.563836,rmse,0.032892,0.014103,0.017040,8,244,0.871797,COMPLETE
2,2,1188.984783,2.073778,0.492736,rmse,8.479396,0.062394,0.006109,19,147,0.463447,COMPLETE
3,3,1318.024755,0.015029,0.764154,rmse,1.176339,0.026305,0.001382,13,145,0.820977,COMPLETE
4,4,1156.852653,0.340799,0.448980,rmse,3.603275,0.536545,0.020737,7,252,0.870824,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...
103,103,1137.453368,5.921774,0.426199,rmse,0.003349,0.097649,0.038925,10,63,0.999080,COMPLETE
104,104,1140.525292,2.501100,0.459475,rmse,0.004910,0.238003,0.034446,11,53,0.918100,COMPLETE
105,105,1135.948779,3.194679,0.526791,rmse,0.014706,0.050586,0.027312,9,77,0.973695,COMPLETE
106,106,1137.280215,3.103511,0.530974,rmse,0.014716,0.050670,0.017724,9,72,0.935026,COMPLETE


In [14]:
df.value.min()

np.float64(1135.1997551744464)

In [15]:
study.best_params

{'lambda': 0.08872320296365035,
 'alpha': 2.505669826820517,
 'gamma': 0.04208748837267979,
 'colsample_bytree': 0.4289384418923935,
 'subsample': 0.9183536581671778,
 'learning_rate': 0.018237495887598765,
 'max_depth': 12,
 'min_child_weight': 84,
 'eval_metric': 'rmse'}

In [ ]:
test = pd.read_csv(f'../input/xg')

In [27]:
%%time
train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

0
1
2
3
4
CPU times: user 2min 26s, sys: 1min 15s, total: 3min 42s
Wall time: 33.3 s


In [28]:
mean_absolute_error(target0, train_oof)

1135.6467608616847

In [30]:
%%time
num_round = 2500


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.01

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.742359873487
CPU times: user 5min 57s, sys: 3min 9s, total: 9min 7s
Wall time: 1min 21s


In [31]:
%%time
num_round = 2700


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.01

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.6704479178497
CPU times: user 6min 18s, sys: 3min 24s, total: 9min 43s
Wall time: 1min 27s


In [32]:
%%time
num_round = 3000


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.01

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.6667403679148
CPU times: user 6min 55s, sys: 3min 47s, total: 10min 43s
Wall time: 1min 35s


In [33]:
%%time
num_round = 6000


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.005

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.3884663044919
CPU times: user 13min 47s, sys: 7min 35s, total: 21min 23s
Wall time: 3min 11s


In [35]:
%%time
num_round = 5000


train_oof = np.zeros((target.shape[0],))
test_preds = 0
kf = KFold(5, shuffle=True, random_state=137)

params = study.best_params

params['learning_rate'] = 0.005

for i, (train_index, val_index) in enumerate(kf.split(train,target)):
    print(i)
    dtrain = xgb.DMatrix(train_folds[i].values, train_ys[i], enable_categorical=True)
    dval = xgb.DMatrix(val_folds[i].values, val_ys[i], enable_categorical=True)
        
    output = xgb.train(params, dtrain, num_round)

    oof_predictions = output.predict(dval)
    train_oof[val_index] = np.exp(oof_predictions) - shift

print(mean_absolute_error(target0, train_oof))

0
1
2
3
4
1134.4089241056433
CPU times: user 11min 46s, sys: 6min 19s, total: 18min 5s
Wall time: 2min 42s


In [36]:
x_test = pd.read_csv('../input/x_test_l.csv')

In [39]:
np.exp(output.predict(xgb.DMatrix(x_test))) - shift

array([2802.0037, 4143.8706, 3017.079 , ..., 4714.319 , 1551.7915,
       2243.5032], shape=(125546,), dtype=float32)